In [33]:
import pandas as pd
import numpy as np
import os

In [34]:
print(os.getcwd())

C:\Users\acost\Documents\proyecto_analisis\Proyecto-Final-Analisis-de-datos\ipynb


In [35]:
# Carpetas de entrada (datos sucios) y salida (datos limpios)
carpeta_entrada = os.path.join("..", "datasets", "datos_simulados")
carpeta_salida = os.path.join("..", "datasets", "datos_simulados_limpios")
os.makedirs(carpeta_salida, exist_ok=True)

# Aqui vamos guardando las metricas de antes/despues de cada dataset
reporte_calidad = []

In [36]:
################################## TICKETS #######################################

In [37]:
# Cargar y unir las 2 partes de tickets
parte1 = pd.read_csv(os.path.join(carpeta_entrada, "tickets_parte1.csv"))
parte2 = pd.read_csv(os.path.join(carpeta_entrada, "tickets_parte2.csv"))
tickets_df = pd.concat([parte1, parte2], ignore_index=True)
tickets_df

,ID_Ticket,Fecha_Compra,ID_Evento,Precio_USD,Categoria
0,1,2024-08-08,223,-150.00,Prensa
1,2,2024-07-29,407,395.92,General
2,3,2024-07-15,170,379.41,GENERAL
3,4,2024-08-12,448,389.18,v.i.p
4,5,2024-07-08,415,271.01,VIP
...,...,...,...,...,...
1009995,408903,2024-07-10,110,256.95,Prensa
1009996,663594,2024-07-20,169,NaN,Prensa
1009997,727417,2024-08-08,152,NaN,General
1009998,301105,2024-07-26,493,340.70,General


In [38]:
# Medir calidad ANTES de limpiar
filas_antes = len(tickets_df)
nulos_antes = int(tickets_df.isna().sum().sum())
duplicados_antes = int(tickets_df.duplicated().sum())
reporte_calidad.append({"Dataset": "Tickets", "Etapa": "Antes", "Filas": filas_antes, "Nulos": nulos_antes, "Duplicados": duplicados_antes})
print(filas_antes, nulos_antes, duplicados_antes)

1010000 49431 10000


In [39]:
# Quitar filas duplicadas (mismo ID_Ticket repetido)
tickets_df = tickets_df.drop_duplicates(subset="ID_Ticket").copy()

In [40]:
# Arreglar el texto de Categoria (quitar espacios, mayusculas, unificar V.I.P)
tickets_df["Categoria"] = tickets_df["Categoria"].str.strip().str.upper()
tickets_df["Categoria"] = tickets_df["Categoria"].replace("V.I.P", "VIP")
tickets_df["Categoria"].value_counts()

Categoria
GENERAL    343309
VIP        342913
PRENSA     313778
Name: count, dtype: int64

In [41]:
# Un precio negativo no puede existir, se marca como vacio
tickets_df.loc[tickets_df["Precio_USD"] < 0, "Precio_USD"] = np.nan

In [42]:
# Rellenar los precios vacios con la mediana
tickets_df["Precio_USD"] = tickets_df["Precio_USD"].fillna(tickets_df["Precio_USD"].median())

In [43]:
# Medir calidad DESPUES de limpiar
filas_despues = len(tickets_df)
nulos_despues = int(tickets_df.isna().sum().sum())
duplicados_despues = int(tickets_df.duplicated().sum())
reporte_calidad.append({"Dataset": "Tickets", "Etapa": "Despues", "Filas": filas_despues, "Nulos": nulos_despues, "Duplicados": duplicados_despues})
print(filas_despues, nulos_despues, duplicados_despues)

1000000 0 0


In [44]:
# Guardar tickets limpios
tickets_df.to_csv(os.path.join(carpeta_salida, "tickets_limpios.csv"), index=False)

In [45]:
################################## BIOMETRIA #######################################

In [46]:
# Cargar y unir los 4 chunks de biometria
chunk1 = pd.read_json(os.path.join(carpeta_entrada, "biometria_chunk_1.json"))
chunk2 = pd.read_json(os.path.join(carpeta_entrada, "biometria_chunk_2.json"))
chunk3 = pd.read_json(os.path.join(carpeta_entrada, "biometria_chunk_3.json"))
chunk4 = pd.read_json(os.path.join(carpeta_entrada, "biometria_chunk_4.json"))
biometria_df = pd.concat([chunk1, chunk2, chunk3, chunk4], ignore_index=True)
biometria_df

,ID_Lectura,ID_Atleta,BPM,Saturacion_O2
0,1,3872,129.0,97.911995
1,2,2354,133.0,95.272773
2,3,2018,139.0,NaN
3,4,1219,121.0,NaN
4,5,3074,150.0,93.413876
...,...,...,...,...
999995,999996,2932,154.0,96.543665
999996,999997,2442,158.0,99.151234
999997,999998,4903,128.0,93.548586
999998,999999,2634,125.0,97.141152


In [47]:
# Medir calidad ANTES de limpiar
filas_antes = len(biometria_df)
nulos_antes = int(biometria_df.isna().sum().sum())
duplicados_antes = int(biometria_df.duplicated().sum())
reporte_calidad.append({"Dataset": "Biometria", "Etapa": "Antes", "Filas": filas_antes, "Nulos": nulos_antes, "Duplicados": duplicados_antes})
print(filas_antes, nulos_antes, duplicados_antes)

1000000 60000 0


In [48]:
# BPM a veces viene como texto ("Error_Sensor"), lo convertimos a numero
# lo que no se puede convertir queda vacio automaticamente
biometria_df["BPM"] = pd.to_numeric(biometria_df["BPM"], errors="coerce")

In [49]:
# Un BPM fuera de 40-220 no es realista en un humano, se marca como vacio
biometria_df.loc[(biometria_df["BPM"] < 40) | (biometria_df["BPM"] > 220), "BPM"] = np.nan

In [50]:
# Rellenar BPM y Saturacion_O2 vacios con la mediana
biometria_df["BPM"] = biometria_df["BPM"].fillna(biometria_df["BPM"].median())
biometria_df["Saturacion_O2"] = biometria_df["Saturacion_O2"].fillna(biometria_df["Saturacion_O2"].median())

In [51]:
# Medir calidad DESPUES de limpiar
filas_despues = len(biometria_df)
nulos_despues = int(biometria_df.isna().sum().sum())
duplicados_despues = int(biometria_df.duplicated().sum())
reporte_calidad.append({"Dataset": "Biometria", "Etapa": "Despues", "Filas": filas_despues, "Nulos": nulos_despues, "Duplicados": duplicados_despues})
print(filas_despues, nulos_despues, duplicados_despues)

1000000 0 0


In [52]:
# Guardar biometria limpia
biometria_df.to_json(os.path.join(carpeta_salida, "biometria_limpia.json"), orient="records")

In [53]:
################################## MOVILIDAD #######################################

In [54]:
# Cargar movilidad urbana
movilidad_df = pd.read_parquet(os.path.join(carpeta_entrada, "movilidad_urbana.parquet"))
movilidad_df

,ID_Viaje,Ciudad_Sede,Estacion,Pasajeros_Hora
0,1,Lyon,Estadio,9844
1,2,Paris,Aeropuerto,4542
2,3,Paris,Gare du Nord,3408
3,4,Niza,Estadio,4130
4,5,Lyon,Estadio,8179
...,...,...,...,...
999995,999996,None,Aeropuerto,5140
999996,999997,Marsella,Estadio,937
999997,999998,Marsella,Estadio,5459
999998,999999,Lyon,Villa Olimpica,1579


In [55]:
# Medir calidad ANTES de limpiar
filas_antes = len(movilidad_df)
nulos_antes = int(movilidad_df.isna().sum().sum())
duplicados_antes = int(movilidad_df.duplicated().sum())
reporte_calidad.append({"Dataset": "Movilidad", "Etapa": "Antes", "Filas": filas_antes, "Nulos": nulos_antes, "Duplicados": duplicados_antes})
print(filas_antes, nulos_antes, duplicados_antes)

1000000 50000 0


In [56]:
# Arreglar mayusculas/minusculas de Ciudad_Sede ("paRis" -> "Paris", "LYON" -> "Lyon")
movilidad_df["Ciudad_Sede"] = movilidad_df["Ciudad_Sede"].str.strip().str.title()
movilidad_df["Ciudad_Sede"].value_counts()

Ciudad_Sede
Lyon        256913
Paris       255054
Marsella    219362
Niza        218671
Name: count, dtype: int64

In [57]:
# Rellenar ciudades vacias con la ciudad mas comun
ciudad_mas_comun = movilidad_df["Ciudad_Sede"].mode()[0]
movilidad_df["Ciudad_Sede"] = movilidad_df["Ciudad_Sede"].fillna(ciudad_mas_comun)

In [58]:
# La estacion "Estacion_Desconocida_99" no aporta info, se renombra
movilidad_df["Estacion"] = movilidad_df["Estacion"].replace("Estacion_Desconocida_99", "Desconocida")

In [59]:
# Medir calidad DESPUES de limpiar
filas_despues = len(movilidad_df)
nulos_despues = int(movilidad_df.isna().sum().sum())
duplicados_despues = int(movilidad_df.duplicated().sum())
reporte_calidad.append({"Dataset": "Movilidad", "Etapa": "Despues", "Filas": filas_despues, "Nulos": nulos_despues, "Duplicados": duplicados_despues})
print(filas_despues, nulos_despues, duplicados_despues)

1000000 0 0


In [60]:
# Guardar movilidad limpia
movilidad_df.to_parquet(os.path.join(carpeta_salida, "movilidad_limpia.parquet"), index=False)

In [61]:
################################## REPORTE FINAL #######################################

In [62]:
# Tabla comparativa Antes vs Despues, pedida por la rubrica
reporte_df = pd.DataFrame(reporte_calidad)
reporte_df.to_csv(os.path.join(carpeta_salida, "reporte_calidad_antes_despues.csv"), index=False)
reporte_df

,Dataset,Etapa,Filas,Nulos,Duplicados
0,Tickets,Antes,1010000,49431,10000
1,Tickets,Despues,1000000,0,0
2,Biometria,Antes,1000000,60000,0
3,Biometria,Despues,1000000,0,0
4,Movilidad,Antes,1000000,50000,0
5,Movilidad,Despues,1000000,0,0
